# Code for the post https://medium.com/p/3f06d7653a85/edit

Code is from below links with minor changes made for the blog post.

References: 
    https://gist.github.com/karpathy/d4dee566867f8291f086
    
    https://gist.github.com/satyajitvg/9a5f782ccef5ff81f7f9863b62218b06
    
        

In [1]:
"""
Minimal character-level Vanilla RNN model. Written by Andrej Karpathy (@karpathy)
BSD License
"""

'\nMinimal character-level Vanilla RNN model. Written by Andrej Karpathy (@karpathy)\nBSD License\n'

In [ ]:
# Print the input Text
seq_length = 25
data_reader = DataReader("RNN_Example_Input.txt", seq_length)
print(data_reader.data)

In [43]:
import numpy as np


def random_init(num_rows, num_cols):
    return np.random.rand(num_rows, num_cols)*0.01

def zero_init(num_rows, num_cols):
    return np.zeros((num_rows, num_cols))


# To read the training data and make a vocabulary and dictiornary to index the chars
class DataReader:
    def __init__(self, path, seq_length):
        #uncomment below , if you dont want to use any file for text reading and comment next 2 lines
        #self.data = "some really long text to test this. maybe not perfect but should get you going."
        self.fp = open(path, "r")
        self.data = self.fp.read()
        #find unique chars
        chars = list(set(self.data))
        #create dictionary mapping for each char
        self.char_to_ix = {ch:i for (i,ch) in enumerate(chars)}
        self.ix_to_char = {i:ch for (i,ch) in enumerate(chars)}
        #total data
        self.data_size = len(self.data)
        #num of unique chars
        self.vocab_size = len(chars)
        self.pointer = 0
        self.seq_length = seq_length

    def next_batch(self):
        input_start = self.pointer
        input_end = self.pointer + self.seq_length
        inputs = [self.char_to_ix[ch] for ch in self.data[input_start:input_end]]
        targets = [self.char_to_ix[ch] for ch in self.data[input_start+1:input_end+1]]
        self.pointer += self.seq_length
        if self.pointer + self.seq_length + 1 >= self.data_size:
            # reset pointer
            self.pointer = 0
        return inputs, targets

    def just_started(self):
        return self.pointer == 0

    def close(self):
        self.fp.close()


In [13]:
class RNN:
    def __init__(self, hidden_size, vocab_size, seq_length, learning_rate):
        # hyper parameters
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size
        self.seq_length = seq_length
        self.learning_rate = learning_rate
        # model parameters
        self.U = np.random.uniform(-np.sqrt(1./vocab_size), np.sqrt(1./vocab_size), (hidden_size, vocab_size))
        self.V = np.random.uniform(-np.sqrt(1./hidden_size), np.sqrt(1./hidden_size), (vocab_size, hidden_size))
        self.W = np.random.uniform(-np.sqrt(1./hidden_size), np.sqrt(1./hidden_size), (hidden_size, hidden_size))
        self.b = np.zeros((hidden_size, 1)) # bias for hidden layer
        self.c = np.zeros((vocab_size, 1)) # bias for output
        
        # memory vars for adagrad, 
        #ignore if you implement another approach
        self.mU = np.zeros_like(self.U)
        self.mW = np.zeros_like(self.W)
        self.mV = np.zeros_like(self.V)
        self.mb = np.zeros_like(self.b)
        self.mc = np.zeros_like(self.c)

    def softmax(self, x):
        p = np.exp(x- np.max(x))
        return p / np.sum(p)
        
    def forward(self, inputs, hprev):
            xs, hs, os, ycap = {}, {}, {}, {}
            hs[-1] = np.copy(hprev)
            for t in range(len(inputs)):
                xs[t] = zero_init(self.vocab_size,1)
                xs[t][inputs[t]] = 1 # one hot encoding , 1-of-k
                hs[t] = np.tanh(np.dot(self.U,xs[t]) + np.dot(self.W,hs[t-1]) + self.b) # hidden state
                os[t] = np.dot(self.V,hs[t]) + self.c # unnormalised log probs for next char
                ycap[t] = self.softmax(os[t]) # probs for next char
            return xs, hs, ycap
        
    def backward(self, xs, hs, ps, targets):
            # backward pass: compute gradients going backwards
            dU, dW, dV = np.zeros_like(self.U), np.zeros_like(self.W), np.zeros_like(self.V)
            db, dc = np.zeros_like(self.b), np.zeros_like(self.c)
            dhnext = np.zeros_like(hs[0])
            for t in reversed(range(self.seq_length)):
                dy = np.copy(ps[t])
                #through softmax
                dy[targets[t]] -= 1 # backprop into y
                #calculate dV, dc
                dV += np.dot(dy, hs[t].T)
                dc += dc
                #dh includes gradient from two sides, next cell and current output
                dh = np.dot(self.V.T, dy) + dhnext # backprop into h
                # backprop through tanh non-linearity 
                dhrec = (1 - hs[t] * hs[t]) * dh  #dhrec is the term used in many equations
                db += dhrec
                #calculate dU and dW
                dU += np.dot(dhrec, xs[t].T)
                dW += np.dot(dhrec, hs[t-1].T)
                #pass the gradient from next cell to the next iteration.
                dhnext = np.dot(self.W.T, dhrec)
            # clip to mitigate exploding gradients
            for dparam in [dU, dW, dV, db, dc]:
                np.clip(dparam, -5, 5, out=dparam) 
            return dU, dW, dV, db, dc
    
    def loss(self, ps, targets):
            """loss for a sequence"""
            # calculate cross-entrpy loss
            return sum(-np.log(ps[t][targets[t],0]) for t in range(self.seq_length))
        
    
    def update_model(self, dU, dW, dV, db, dc):
        # parameter update with adagrad
        for param, dparam, mem in zip([self.U, self.W, self.V, self.b, self.c],
                                  [dU, dW, dV, db, dc],
                                  [self.mU, self.mW, self.mV, self.mb, self.mc]):
            mem += dparam*dparam
            param += -self.learning_rate*dparam/np.sqrt(mem+1e-8) # adagrad update
                
                
    def sample(self, h, seed_ix, n):
            """
            sample a sequence of integers from the model
            h is memory state, seed_ix is seed letter from the first time step
            """
            x = zero_init(self.vocab_size, 1)
            x[seed_ix] = 1
            ixes = []
            for t in range(n):
                h = np.tanh(np.dot(self.U, x) + np.dot(self.W, h) + self.b)
                y = np.dot(self.V, h) + self.c
                p = np.exp(y)/np.sum(np.exp(y))
                ix = np.random.choice(range(self.vocab_size), p = p.ravel())
                x = zero_init(self.vocab_size,1)
                x[ix] = 1
                ixes.append(ix)
            return ixes

    def train(self, data_reader):
            iter_num = 0
            threshold = 0.01
            smooth_loss = -np.log(1.0/data_reader.vocab_size)*self.seq_length
            while (smooth_loss > threshold):
                if data_reader.just_started():
                    hprev = np.zeros((self.hidden_size,1))
                inputs, targets = data_reader.next_batch()
                xs, hs, ps = self.forward(inputs, hprev)
                dU, dW, dV, db, dc = self.backward(xs, hs, ps, targets)
                loss = self.loss(ps, targets)
                self.update_model(dU, dW, dV, db, dc)
                smooth_loss = smooth_loss*0.999 + loss*0.001
                hprev = hs[self.seq_length-1]
                if not iter_num%500:
                    sample_ix = self.sample(hprev, inputs[0], 200)
                    print( ''.join(data_reader.ix_to_char[ix] for ix in sample_ix))
                    print( "\n\niter :%d, loss:%f"%(iter_num, smooth_loss))
                iter_num += 1

    def predict(self, data_reader, start, n):

        #initialize input vector
        x = zero_init(self.vocab_size, 1)
        chars = [ch for ch in start]
        ixes = []
        for i in range(len(chars)):
            ix = data_reader.char_to_ix[chars[i]]
            x[ix] = 1
            ixes.append(ix)

        h = np.zeros((self.hidden_size,1))
        # predict next n chars
        for t in range(n):
            h = np.tanh(np.dot(self.U, x) + np.dot(self.W, h) + self.b)
            y = np.dot(self.V, h) + self.c
            p = np.exp(y)/np.sum(np.exp(y))
            ix = np.random.choice(range(self.vocab_size), p = p.ravel())
            x = zero_init(self.vocab_size,1)
            x[ix] = 1
            ixes.append(ix)
        txt = ''.join(data_reader.ix_to_char[i] for i in ixes)
        return txt

In [44]:
seq_length = 25
data_reader = DataReader("RNN_Example_Input.txt", seq_length)
print(data_reader.data)

Weather is a fascinating natural phenomenon that influences many aspects of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perceptions of the world. Understanding and predicting the weather has always been a topic of great interest and study for both scientists and the general public. One of the most obvious effects of weather is its impact on the environment. The Earth's climate is the result of long-term weather patterns, which determine the distribution of temperature, precipitation, and wind across the planet. Different regions experience distinct climates, such as the dry desert climate of the Sahara or the cold tundra climate of the Arctic. 


In [46]:
seq_length = 25
#read text from the "input.txt" file
data_reader = DataReader("RNN_Example_Input.txt", seq_length)

# Train the model
rnn = RNN(hidden_size=100, vocab_size=data_reader.vocab_size,seq_length=seq_length,learning_rate=1e-1)
rnn.train(data_reader)

WcfintTiriTSr mcmoeigESntvTbADtSFExurEdmOu fTUO, bsfEbve,Ao,rrximEOf nwxe.eSScpuUtofmausWimFEdyx Sx.WfUEsh f,l.nwtwfnE vof ic,FdvrFWSoFf,fUfOu.  WDhtfvoTA,-OA foelnDdbAAieathT.WF fscUn F osEWs fn,mxF'


iter :0, loss:89.587836
ta , ot tioi tt rco endowpotebo.c fe ctfisss eas  n us terf tuleri bSra i crime thheymnttfist as il Tiles aten. tfsth ttaleeselwt f aemerat oSs oeupc oherEobbagcat. grtonsopipte te ore un,thwT yntett 


iter :500, loss:83.339233
teincthere ietts wael ilecte thetenn dte lentermanm pf pandasds amarong eralendtucricpang windeang erang  xund Eafvexvercteng acic is cheaminctiseSenel wiang of testos of moncg mermessrdeothaooy s wer


iter :1000, loss:72.219317
rad raine Tf the dhes DUy thatio in the ofate ny Dendin ertimal dreeArerare of the ts Drestes th ierathecthe ourmiats r  laacientheshic ine iur ofetey The pl 're orh in tha sranoo. olheang Urd-re tla 


iter :1500, loss:63.675394
t. of tholom terofhaescheingtermendre sm the werch of orcsest wlomathe wurce ofothe 

etiinal ifatests on ofr dis iffation, and wind across the planet. Different regions experience distinct climates, such as the dry desert climate of the Sahara or the cold tundra climate of the Arctict


iter :18000, loss:0.377315
weptha cls fascepest scriss the wiatherimate of the Sahara or the cold tundra climate of the Arcticting the weather has always ing ouvic. iblth'scend rerdint enffatenas iof the dry dhing our experienc


iter :18500, loss:0.355489
ncionsteres. From determin a scteras the dry roimenon the most obvious effects of weather paetiences ote the distribution, pndicl ffromesepas ean, sxmicpecting travel planf, weather infa woferestistin


iter :19000, loss:0.331770
atiral puterol  and tur clinsture, entens, sciompatherimpte os our daily live an isstina ol oure os the resuresting tredere ts oor both ng of -ha planmbutheatspacts actoo impacting travel plans, weath


iter :19500, loss:0.310001
seathe mene s reris the d deres, sciest engerivis pinces mpathesticlimate distin

atiras act restind ace s wert climates, sundraential role in shaping our experiences and perceptions of the world. Understanding and predicting the weather has always been a topic of great interest an


iter :36000, loss:0.147924
s and study for both scientists and the general public. One of the most obvious effects of weather is its impact on the environment. The Earth's climate is the result of long-term weather patterns, wh


iter :36500, loss:0.142713
ncimantes many aspects of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role ing weather pand tundspend ceptions of the world. Understand


iter :37000, loss:0.138134
ur dateras the cold tundra climate of the Arcticts of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and p


iter :37500, loss:0.134058
s boin the dry desert climate of the Sahara or the cold tundral pot climate of t

at rating atiperent on th ice plan turesul ping the weather has always been a topic of great interest and study for both scientists and the general public. One of the most obvious effects of weather i


iter :54000, loss:0.197714
s at aving our experien in s ans alwestind clothend an the geseris the monm our plothing choices to impacting travel plays of the world. Understa in the dry desert climate of the Arcticts of our daily


iter :54500, loss:0.298481
scinfl prenmpatt our experiences and perceptions of the world. Understanding and predicting the weather has always been a topic of great interest and study for bold weather plays an essential role in 


iter :55000, loss:0.252365
at raat in of the dry desert climate of the Sahara or the cold tundra climate of the Arcticts of our daily lives. From determining our clothing choices to impacting travel peatia weather is its impact


iter :55500, loss:0.205467
s at as of the result of long-term weather patterns, which determine the distrib

atural phenomenon that influences many aspects of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perce


iter :72000, loss:0.078252
weytiats of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perceptions of the world. Understanding and


iter :72500, loss:0.078526
. inf pind across end repiclimate is the result of long-term weather is its impact on the environment. The Earth's climate is the result of long-term weather patterns, which determine the distribution


iter :73000, loss:0.077928
atural phenomenon that influences many aspects of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perce


iter :73500, loss:0.076712
s ating natfengians ans of the Sara, precipitation, and wind across the planet. 

...

aterteris and the ceat potiouensean ts teathes is its impact on the envirand of lh. Ungerays orenipanding an phenomenot our ong-ter ss trichisf of the worche on the ervir st of oh ict clon, and wind a


iter :216000, loss:0.015729
weathat ilf uene pfand the enverancl determine the distribution of temperature, precipitation, and wind across the planet. Different regions experience distinct climates, such as the dry desert climat


iter :216500, loss:0.015667
ncions eane the dry duye, our clinftr dy ding on the dry desert climate of the Sahara or the cold tundra climate of the Arctict nnterent perions experience distinct climate is the result of long-term 


iter :217000, loss:0.015601
aterth ge exterila f sceny ling the weather has always been a topic of great interman l intor dite os our daily lives. From determining our clothing choices to impacting travel plans, weather plays an


iter :217500, loss:0.015539
s and surdre oumpethe the dis roim of our daily lives. From determining our 

atimenos iore of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perceptions of the world. Understandin


iter :234000, loss:0.013785
s and surdre theacha f sentistsnd phe of the most obvious effects of weather is its impact on the environment. The Earth's climate is the result of long-term weather patterns, which determine the dist


iter :234500, loss:0.013742
nching the weather has always been a topic of great interest and study for both scientists and the general public. One of the most obvious effects of weather is its impact on the environment. The Eart


iter :235000, loss:0.013694
atural phenomenon that influences many aspects of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perce


iter :235500, loss:0.013651
weatheagenos bheat indes mate is the result of long-term weather patterns, w

emibatitacion os the dry desert climate of the Arctict nature, precipitation, and wind across the planet. Different regions experience distinct climates, such as the dry desert climate of the Sahara o


iter :252000, loss:0.012361
s and surdra climate of the Arctict ncte of the Sahara or the cold tundra climate of the Arcticting the weather has always been a topic of great interest and study for both scientists and the general 


iter :252500, loss:0.012328
ncions eate dist ind predictotien is the desting treverand sund surl of th teethe climate of the Sahara or the cold tundra climate of the Arctict and strdis of the environment. The Earth's climate is 


iter :253000, loss:0.012292
atural phenomenon that influences many aspects of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perce


iter :253500, loss:0.012259
weyth's influences and perceptions of the world. Undertand, ore heserting ta

atural phenomenon that influences many aspects of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perce


iter :270000, loss:0.011252
s ans and surd ans stunding the weather has always been a topic of great interest and study for both scientists and the general public. One of the most obvious effects of weather is its impact on the 


iter :270500, loss:0.011226
nalung tathing-termpaace of the Sahara or the cold tundra climate of the Arcticts of the most on th perats the destices ind the grand suris and pard across the planeturecions eatires on ond pladit ins


iter :271000, loss:0.011197
atural phenomenon that influences many aspects of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perce


iter :271500, loss:0.011170
weatheagenos impacting travel plans, weather plays an essential role in shap

atural phenomenon that influences many aspects of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perce


iter :288000, loss:0.010351
weyth's influences and perd surld and piudet. Ond Arcticts at ofetopice of the Sahara or the cold tundra climate of the Arctict and stoin the environment. The Earth's climate is the result of long-ter


iter :288500, loss:0.010329
ncions envestina the weather is its impact on the environment. The Earth's climate is the result of long-term weather patterns, which determine the distribution of temperature, precipitation, and wind


iter :289000, loss:0.010305
esthe Sof tud Sahara of the most obvious effects of weather is its impact on the environment. The Earth's climate is the result of long-term weather patterns, which determine the distribution of tempe


iter :289500, loss:0.010283
weathe grientists and the general public. One of our daily lives. From deter

In [22]:
# Run the model
rnn.predict(data_reader, 'get', 50)

'getrompttr bol weather is its impact on the environme'

In [19]:
# Understand various data structures used to store results
fp = open("RNN_Example_Input.txt", "r")
dd = fp.read()
#find unique chars
print("Data read: " + dd)
chars = list(set(dd))
chars
#create dictionary mapping for each char
char_to_ix = {ch:i for (i,ch) in enumerate(chars)}
ix_to_char = {i:ch for (i,ch) in enumerate(chars)}
#total data
data_size = len(dd)
#num of unique chars
vocab_size = len(chars)
pointer = 0
seq_length = seq_length

Data read: Weather is a fascinating natural phenomenon that influences many aspects of our daily lives. From determining our clothing choices to impacting travel plans, weather plays an essential role in shaping our experiences and perceptions of the world. Understanding and predicting the weather has always been a topic of great interest and study for both scientists and the general public. One of the most obvious effects of weather is its impact on the environment. The Earth's climate is the result of long-term weather patterns, which determine the distribution of temperature, precipitation, and wind across the planet. Different regions experience distinct climates, such as the dry desert climate of the Sahara or the cold tundra climate of the Arctic. 


In [10]:
pointer = 0
seq_length = 25
input_start = pointer
input_end = pointer + seq_length
inputs = [char_to_ix[ch] for ch in dd[input_start:input_end]]
targets = [char_to_ix[ch] for ch in dd[input_start+1:input_end+1]]
print("inputs: " + str(inputs))
print("targets: " + str(targets))
pointer += seq_length
print("pointer: " + str(pointer))

inputs: [7, 19, 21, 28, 34, 19, 3, 8, 14, 9, 8, 21, 8, 33, 21, 9, 1, 14, 6, 21, 28, 14, 6, 20, 8]
targets: [19, 21, 28, 34, 19, 3, 8, 14, 9, 8, 21, 8, 33, 21, 9, 1, 14, 6, 21, 28, 14, 6, 20, 8, 6]
pointer: 25


In [11]:
# Print the dimensions of various matrices and vectors

print("length of inputs: " + str(len(inputs)))
print("length of targets: " + str(len(targets)))
hidden_size = 100
vocab_size = len(chars)
print("hidden_size: " + str(hidden_size))
print("vocab_size: " + str(vocab_size))
print("seq_length: " + str(seq_length))
print("learning_rate: " + str(1e-1))
# model parameters
U = np.random.uniform(-np.sqrt(1./vocab_size), np.sqrt(1./vocab_size), (hidden_size, vocab_size))
V = np.random.uniform(-np.sqrt(1./hidden_size), np.sqrt(1./hidden_size), (vocab_size, hidden_size))
W = np.random.uniform(-np.sqrt(1./hidden_size), np.sqrt(1./hidden_size), (hidden_size, hidden_size))
b = np.zeros((hidden_size, 1)) # bias for hidden layer
c = np.zeros((vocab_size, 1)) # bias for output
print("Dimension of U: " + str(np.shape(U)))
print("Dimension of V: " + str(np.shape(V)))
print("Dimension of W: " + str(np.shape(W)))
print("Dimension of b: " + str(np.shape(b)))
print("Dimension of c: " + str(np.shape(c)))

length of inputs: 25
length of targets: 25
hidden_size: 100
vocab_size: 36
seq_length: 25
learning_rate: 0.1
Dimension of U: (100, 36)
Dimension of V: (36, 100)
Dimension of W: (100, 100)
Dimension of b: (100, 1)
Dimension of c: (36, 1)


In [ ]:
# Code below is to understand and follow through different data structures and data operations
# You can ignore it

print("Code below is to understand and follow through different data structures and data operations You can ignore it")

In [89]:
print(set(dd))

{'O', 'a', 'u', 'e', 'v', 'm', 'n', 'T', ' ', 'U', 'A', "'", 'S', 't', 'W', 'i', 'h', '.', 'x', 'r', 'd', 'g', '-', 'w', 'c', 's', 'E', 'l', 'p', 'f', 'o', ',', 'y', 'F', 'D', 'b'}


In [90]:
print(list(set(dd)))

['O', 'a', 'u', 'e', 'v', 'm', 'n', 'T', ' ', 'U', 'A', "'", 'S', 't', 'W', 'i', 'h', '.', 'x', 'r', 'd', 'g', '-', 'w', 'c', 's', 'E', 'l', 'p', 'f', 'o', ',', 'y', 'F', 'D', 'b']


In [17]:
{ch:i for (i,ch) in enumerate(chars)}

{'O': 0,
 'a': 1,
 'u': 2,
 'e': 3,
 'v': 4,
 'm': 5,
 'n': 6,
 'T': 7,
 ' ': 8,
 'U': 9,
 'A': 10,
 "'": 11,
 'S': 12,
 't': 13,
 'W': 14,
 'i': 15,
 'h': 16,
 '.': 17,
 'x': 18,
 'r': 19,
 'd': 20,
 'g': 21,
 '-': 22,
 'w': 23,
 'c': 24,
 's': 25,
 'E': 26,
 'l': 27,
 'p': 28,
 'f': 29,
 'o': 30,
 ',': 31,
 'y': 32,
 'F': 33,
 'D': 34,
 'b': 35}

In [27]:
print({ch:i for (i,ch) in list(enumerate(set("Hello")))})

{'H': 0, 'l': 1, 'o': 2, 'e': 3}


In [28]:
{i:ch for (i,ch) in enumerate(chars)}

{0: 'O',
 1: 'a',
 2: 'u',
 3: 'e',
 4: 'v',
 5: 'm',
 6: 'n',
 7: 'T',
 8: ' ',
 9: 'U',
 10: 'A',
 11: "'",
 12: 'S',
 13: 't',
 14: 'W',
 15: 'i',
 16: 'h',
 17: '.',
 18: 'x',
 19: 'r',
 20: 'd',
 21: 'g',
 22: '-',
 23: 'w',
 24: 'c',
 25: 's',
 26: 'E',
 27: 'l',
 28: 'p',
 29: 'f',
 30: 'o',
 31: ',',
 32: 'y',
 33: 'F',
 34: 'D',
 35: 'b'}

In [47]:
np.shape(U)

(100, 36)

In [109]:
mm = {}
mm

{}

In [112]:
for i in np.arange(3):
    mm[i] = np.zeros((5,1))
mm

{0: array([[0.],
        [0.],
        [0.],
        [0.],
        [0.]]),
 1: array([[0.],
        [0.],
        [0.],
        [0.],
        [0.]]),
 2: array([[0.],
        [0.],
        [0.],
        [0.],
        [0.]])}

In [47]:
np.shape(U)

(100, 36)

In [109]:
mm = {}
mm

{}

In [112]:
for i in np.arange(3):
    mm[i] = np.zeros((5,1))
mm

{0: array([[0.],
        [0.],
        [0.],
        [0.],
        [0.]]),
 1: array([[0.],
        [0.],
        [0.],
        [0.],
        [0.]]),
 2: array([[0.],
        [0.],
        [0.],
        [0.],
        [0.]])}

In [24]:
U

array([[-0.1574878 ,  0.1042762 ,  0.12213002, ...,  0.07441198,
        -0.01356703, -0.10877133],
       [ 0.09208198,  0.1549365 ,  0.05205793, ...,  0.11210908,
        -0.01229183, -0.10667555],
       [ 0.07353221, -0.10427   , -0.02596711, ..., -0.08665776,
         0.11744734, -0.10181764],
       ...,
       [ 0.03583948,  0.0019824 , -0.04006142, ...,  0.15658534,
         0.13511528,  0.05841551],
       [-0.15387769,  0.03392269, -0.07807148, ..., -0.13696114,
         0.04084846,  0.12465368],
       [ 0.13659038, -0.09998326,  0.02645249, ...,  0.07639554,
        -0.05238045, -0.07926685]])

In [26]:
np.random.uniform(-np.sqrt(1./vocab_size), np.sqrt(1./vocab_size), (hidden_size, vocab_size))

array([[ 0.020854  , -0.15099593,  0.10247701, ...,  0.04988185,
        -0.02568777, -0.10721446],
       [ 0.09447984, -0.10378234, -0.07013593, ..., -0.0794748 ,
        -0.12287187,  0.15233802],
       [-0.11671704, -0.00508229, -0.1545735 , ...,  0.13168042,
         0.09361749,  0.05463034],
       ...,
       [-0.00786977, -0.12730455, -0.12299597, ..., -0.06614414,
         0.06681259,  0.1021587 ],
       [-0.1203155 ,  0.11316926,  0.11810473, ...,  0.16616681,
         0.12825062, -0.0141287 ],
       [ 0.10810506, -0.16415954, -0.16282675, ...,  0.10426053,
         0.15710204, -0.11171012]])

In [34]:
[U, W, V]

[array([[-0.1574878 ,  0.1042762 ,  0.12213002, ...,  0.07441198,
         -0.01356703, -0.10877133],
        [ 0.09208198,  0.1549365 ,  0.05205793, ...,  0.11210908,
         -0.01229183, -0.10667555],
        [ 0.07353221, -0.10427   , -0.02596711, ..., -0.08665776,
          0.11744734, -0.10181764],
        ...,
        [ 0.03583948,  0.0019824 , -0.04006142, ...,  0.15658534,
          0.13511528,  0.05841551],
        [-0.15387769,  0.03392269, -0.07807148, ..., -0.13696114,
          0.04084846,  0.12465368],
        [ 0.13659038, -0.09998326,  0.02645249, ...,  0.07639554,
         -0.05238045, -0.07926685]]),
 array([[-0.06022002, -0.05917283,  0.09110111, ...,  0.04498076,
         -0.06017466, -0.0143414 ],
        [ 0.01948962, -0.00996043,  0.02139822, ..., -0.05964811,
          0.04925471, -0.04842909],
        [-0.01750069, -0.07607441, -0.05506093, ...,  0.02781475,
         -0.04427494, -0.04044982],
        ...,
        [ 0.07433891, -0.04472283,  0.08044468, ..., -

In [35]:
#mem += dparam*dparam
U*U

array([[2.48024061e-02, 1.08735268e-02, 1.49157428e-02, ...,
        5.53714299e-03, 1.84064303e-04, 1.18312031e-02],
       [8.47909115e-03, 2.40053199e-02, 2.71002767e-03, ...,
        1.25684456e-02, 1.51089072e-04, 1.13796740e-02],
       [5.40698612e-03, 1.08722330e-02, 6.74290585e-04, ...,
        7.50956683e-03, 1.37938783e-02, 1.03668322e-02],
       ...,
       [1.28446839e-03, 3.92989920e-06, 1.60491733e-03, ...,
        2.45189698e-02, 1.82561388e-02, 3.41237195e-03],
       [2.36783433e-02, 1.15074909e-03, 6.09515657e-03, ...,
        1.87583544e-02, 1.66859698e-03, 1.55385405e-02],
       [1.86569317e-02, 9.99665321e-03, 6.99734407e-04, ...,
        5.83627911e-03, 2.74371181e-03, 6.28323284e-03]])

In [40]:
np.shape(U*U)

(100, 36)

In [42]:
U/np.sqrt(U*U+1e-8)

array([[-0.9999998 ,  0.99999954,  0.99999966, ...,  0.9999991 ,
        -0.99997284, -0.99999958],
       [ 0.99999941,  0.99999979,  0.99999816, ...,  0.9999996 ,
        -0.99996691, -0.99999956],
       [ 0.99999908, -0.99999954, -0.99999258, ..., -0.99999933,
         0.99999964, -0.99999952],
       ...,
       [ 0.99999611,  0.99873013, -0.99999688, ...,  0.9999998 ,
         0.99999973,  0.99999853],
       [-0.99999979,  0.99999566, -0.99999918, ..., -0.99999973,
         0.999997  ,  0.99999968],
       [ 0.99999973, -0.9999995 ,  0.99999285, ...,  0.99999914,
        -0.99999818, -0.9999992 ]])